In [1]:
import spacy
import benepar
from tqdm import tqdm
import json

In [8]:

nlp = spacy.load('en_core_web_md')
nlp.add_pipe('benepar', config={'model': 'benepar_en3'})

# Read your file
file_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data/train.txt"
with open(file_path, 'r') as f:
    lines = f.readlines()

# Parse each line
results = []
for i, line in enumerate(tqdm(lines)):
    line = line.strip()
    if line:
        doc = nlp(line)
        for sent in doc.sents:
            results.append({
                'line_number': i,
                'text': sent.text,
                'parse_tree': sent._.parse_string
            })

# Save results
with open('parsed_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"Parsed {len(results)} sentences and saved to parsed_results.json")
# The time for action

LookupError: 
**********************************************************************
  Resource [93mbenepar_en3[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> benepar.download('benepar_en3')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mmodels/benepar_en3[0m

  Searched in:
    - '/home/mrenaudin/nltk_data'
    - '/home/mrenaudin/.conda/envs/leaps3/nltk_data'
    - '/home/mrenaudin/.conda/envs/leaps3/share/nltk_data'
    - '/home/mrenaudin/.conda/envs/leaps3/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


In [5]:
import spacy
import benepar
from tqdm import tqdm
import json

nlp = spacy.load('en_core_web_md')
nlp.add_pipe('benepar', config={'model': 'benepar_en3'})

file_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data/train.txt"
output_file = 'parsed_results.json'

batch_size = 1000
write_every = 10000  # Write to file every 10k sentences

results = []
total_sentences = 0

with open(file_path, 'r') as f:
    batch = []
    
    for line_num, line in enumerate(tqdm(f, desc="Processing")):
        line = line.strip()
        if line:
            batch.append((line_num, line))
            
            if len(batch) == batch_size:
                # Process batch
                texts = [item[1] for item in batch]
                docs = list(nlp.pipe(texts))
                
                for (line_idx, _), doc in zip(batch, docs):
                    for sent in doc.sents:
                        results.append({
                            'line_number': line_idx,
                            'text': sent.text,
                            'parse_tree': sent._.parse_string
                        })
                
                total_sentences += len(results)
                
                # Write and clear memory every 10k sentences
                if len(results) >= write_every:
                    # Append to file
                    mode = 'w' if total_sentences == len(results) else 'a'
                    with open(output_file, mode) as out_f:
                        if mode == 'w':
                            out_f.write('[\n')
                        for i, result in enumerate(results):
                            if total_sentences > len(results) or i > 0:
                                out_f.write(',\n')
                            json.dump(result, out_f)
                    
                    results = []  # Clear memory
                
                batch = []

# Handle remaining items
if batch:
    texts = [item[1] for item in batch]
    docs = list(nlp.pipe(texts))
    for (line_idx, _), doc in zip(batch, docs):
        for sent in doc.sents:
            results.append({
                'line_number': line_idx,
                'text': sent.text,
                'parse_tree': sent._.parse_string
            })

# Write final batch
if results:
    with open(output_file, 'a') as out_f:
        if total_sentences > 0:
            out_f.write(',\n')
        for i, result in enumerate(results):
            if i > 0:
                out_f.write(',\n')
            json.dump(result, out_f)

# Close JSON array
with open(output_file, 'a') as out_f:
    out_f.write('\n]')

print(f"Parsing complete. Total sentences: {total_sentences + len(results)}")

Processing: 0it [00:00, ?it/s]You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Processing: 14999it [12:29, 20.00it/s]


KeyboardInterrupt: 

In [6]:
import spacy
import benepar
from tqdm import tqdm

nlp = spacy.load('en_core_web_sm')  # Smaller model
nlp.add_pipe('benepar', config={'model': 'benepar_en3'})

# Disable unnecessary components for speed
nlp.disable_pipes(['ner', 'lemmatizer'])

file_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data/train.txt"

batch_size = 5000  # Much larger batches
line_count = 0

with open('parsed_results.txt', 'w') as out_f:  # Plain text, not JSON
    with open(file_path, 'r') as in_f:
        batch = []
        
        for line in tqdm(in_f):
            line = line.strip()
            if line:
                batch.append(line)
                
                if len(batch) == batch_size:
                    # Process entire batch at once
                    for doc in nlp.pipe(batch, batch_size=batch_size):
                        for sent in doc.sents:
                            # Write immediately, don't store in memory
                            out_f.write(f"{sent.text}\t{sent._.parse_string}\n")
                    
                    batch = []
                    line_count += batch_size
        
        # Process remaining
        if batch:
            for doc in nlp.pipe(batch):
                for sent in doc.sents:
                    out_f.write(f"{sent.text}\t{sent._.parse_string}\n")

print(f"Done. Output in parsed_results.txt")

0it [00:00, ?it/s]You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
4999it [00:25, 198.05it/s]


KeyboardInterrupt: 

In [16]:
from datasets import load_dataset
from transformers import pipeline
import numpy as np

# Load LLM
llm = pipeline("text-generation", model="distilgpt2")

# BLiMP datasets for our target structures
datasets = {
    "wh_questions": "wh_questions_object_gap",
    "relative_clauses": "relative_clause_object", 
    "passives": "passive_1"
}

def ask_llm(sentence, structure_type):
    prompt = f"Does this sentence contain {structure_type}? Answer yes or no: '{sentence}'"
    response = llm(prompt, max_new_tokens=5, do_sample=False)[0]['generated_text']
    return 'yes' in response.lower()

# Evaluate each dataset
results = {}

for name, dataset_name in datasets.items():
    print(f"Evaluating {name}...")
    
    # Load dataset
    data = load_dataset("blimp", dataset_name)['train']
    data = data['sentence_good']
    print('data', data)
    correct = 0
    total = 0
    
    for sentence in data[:100]:  # First 100 examples
        
        # Ask LLM if it finds the structure
        llm_found = ask_llm(sentence, name.replace('_', ' '))
        
        # BLiMP: if sentence_good exists, structure should be present
        ground_truth = True  # Grammatical sentences should have the structure
        
        if llm_found == ground_truth:
            correct += 1
        
        total += 1
    
    accuracy = correct / total
    results[name] = accuracy
    print(f"{name}: {accuracy:.3f}")

print("\nFinal Results:")
for name, acc in results.items():
    print(f"{name}: {acc:.3f}")

Device set to use cpu


Evaluating wh_questions...


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


data ['Joel discovered the vase that Patricia might take.', 'Rodney thought about many paintings that Wendy had thought about.', "Susan wasn't thinking about the rivers that Kenneth wasn't bringing.", 'A driver has thought about those deer that many pictures look like.', 'Amy has seen every skateboard that all guests see.', 'Michelle can see this banana that Maria did sell.', 'Rachelle thought about a screen that Galileo hides.', 'These waitresses see some mirrors that Phillip found.', "Pamela thought about most students that a lot of libraries haven't praised.", 'Mark discovers most analyses that Randolf remembered.', 'Caroline forgets most customers that Michelle has admired.', "Bill wasn't noticing the pedestrian that Joseph wasn't talking to.", 'Teresa knew that man that April remembered.', 'This boy knew a lot of pedestrians that the Clintons watch.', 'Most patients investigate most patients that Sonia was insulting.', 'Carlos noticed a lot of eggplants that these doctors find.', 

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:5

wh_questions: 1.000
Evaluating relative_clauses...


ValueError: BuilderConfig 'relative_clause_object' not found. Available: ['adjunct_island', 'anaphor_gender_agreement', 'anaphor_number_agreement', 'animate_subject_passive', 'animate_subject_trans', 'causative', 'complex_NP_island', 'coordinate_structure_constraint_complex_left_branch', 'coordinate_structure_constraint_object_extraction', 'determiner_noun_agreement_1', 'determiner_noun_agreement_2', 'determiner_noun_agreement_irregular_1', 'determiner_noun_agreement_irregular_2', 'determiner_noun_agreement_with_adj_2', 'determiner_noun_agreement_with_adj_irregular_1', 'determiner_noun_agreement_with_adj_irregular_2', 'determiner_noun_agreement_with_adjective_1', 'distractor_agreement_relational_noun', 'distractor_agreement_relative_clause', 'drop_argument', 'ellipsis_n_bar_1', 'ellipsis_n_bar_2', 'existential_there_object_raising', 'existential_there_quantifiers_1', 'existential_there_quantifiers_2', 'existential_there_subject_raising', 'expletive_it_object_raising', 'inchoative', 'in

In [21]:
from datasets import load_dataset
from transformers import pipeline
import re

# Load LLM
llm = pipeline("text-generation", model="google/flan-t5-small")

target_datasets = {
    "wh_questions": "wh_questions_object_gap",
    "passives": "passive_1", 
    "complex_np": "complex_np_islands"
}
control_dataset = "determiner_noun_agreement_1"

def ask_llm(sentence):
    prompt = f"""Classify this sentence into ONE category:

    Sentence: "{sentence}"

    Categories:
    - wh-questions: Contains question words (what, who, where, when, why, how) or question structure
    - passive: Contains passive voice (subject receives action, often with "by")
    - complex-np: Contains complex noun phrases or embedded structures
    - none: Simple declarative sentence with none of the above

    Think: What is the main syntactic feature?

    Classification:"""    
    response = llm(prompt, max_new_tokens=5, do_sample=False)[0]['generated_text']
    response_clean = response.replace(prompt, "").strip().lower()
    
    # Parse response
    if 'wh' in response_clean:
        return 'wh_questions'
    elif 'passive' in response_clean:
        return 'passives'
    elif 'complex' in response_clean:
        return 'complex_np'
    else:
        return 'none'
    
def evaluate_dataset(dataset_name, expected_answer, name):
    print(f"\n=== {name.upper()} (expecting: {expected_answer}) ===")
    data = load_dataset("blimp", dataset_name)['train']['sentence_good']
    
    correct = 0
    total = 20  # Test on 20 examples
    
    for i, sentence in enumerate(data[:total]):
        llm_answer = ask_llm(sentence)
        
        is_correct = llm_answer == expected_answer
        if is_correct:
            correct += 1
        
        print(f"{i+1:2d}. {sentence[:40]}...")
        print(f"    LLM: {llm_answer} | Expected: {expected_answer} | {'✓' if is_correct else '✗'}")
    
    accuracy = correct / total
    print(f"\nAccuracy: {accuracy:.3f}")
    return accuracy

# Evaluate target datasets
results = {}
for name, dataset_name in target_datasets.items():
    results[name] = evaluate_dataset(dataset_name, name, name)

# Evaluate control dataset (should be "none")
results['control'] = evaluate_dataset(control_dataset, 'none', 'control')

# Summary
print(f"\n{'='*50}")
print("FINAL RESULTS:")
print(f"{'='*50}")
for name, acc in results.items():
    print(f"{name:15}: {acc:.3f}")

overall = sum(results.values()) / len(results)
print(f"{'Overall':15}: {overall:.3f}")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Device set to use cpu
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2Fo


=== WH_QUESTIONS (expecting: wh_questions) ===


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


 1. Joel discovered the vase that Patricia m...
    LLM: none | Expected: wh_questions | ✗
 2. Rodney thought about many paintings that...
    LLM: none | Expected: wh_questions | ✗
 3. Susan wasn't thinking about the rivers t...
    LLM: none | Expected: wh_questions | ✗
 4. A driver has thought about those deer th...
    LLM: none | Expected: wh_questions | ✗
 5. Amy has seen every skateboard that all g...
    LLM: none | Expected: wh_questions | ✗
 6. Michelle can see this banana that Maria ...
    LLM: none | Expected: wh_questions | ✗


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


...

 6. Who have most people discovered while em...
    LLM: none | Expected: none | ✓
 7. Who was William firing before talking ab...
    LLM: none | Expected: none | ✓
 8. What is Denise descending while hiding a...
    LLM: none | Expected: none | ✓
 9. Who does John leave while alarming Bever...
    LLM: none | Expected: none | ✓
10. What was Melanie going to after taking t...
    LLM: none | Expected: none | ✓


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


11. Who could Bethany research before discus...
    LLM: none | Expected: none | ✓
12. What could Jessica sell before noticing ...
    LLM: none | Expected: none | ✓
13. What had Helen biked to without botherin...
    LLM: none | Expected: none | ✓
14. Who is Mary irritating after approaching...
    LLM: none | Expected: none | ✓
15. Who might Rose flee from before returnin...
    LLM: none | Expected: none | ✓


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


16. What will Janice research after boasting...
    LLM: none | Expected: none | ✓
17. Who had Karla aggravated without finding...
    LLM: none | Expected: none | ✓
18. Who should a government reference before...
    LLM: none | Expected: none | ✓
19. What had Aaron sounded like while cleani...
    LLM: none | Expected: none | ✓
20. What is Brenda arriving at while exiting...
    LLM: none | Expected: none | ✓

Accuracy: 1.000

FINAL RESULTS:
wh_questions   : 0.000
passives       : 0.000
control        : 1.000
Overall        : 0.333


In [23]:
from datasets import load_dataset
from transformers import pipeline

# Load reasoning model
llm = pipeline("text-generation", model="distilgpt2")

# BLiMP datasets
target_datasets = {
    "wh_questions": "wh_questions_object_gap",
    "passives": "passive_1", 
    "complex_np": "complex_np_islands"
}
control_dataset = "determiner_noun_agreement_1"

def ask_llm_reasoning(sentence):
    prompt = f"""Classify this sentence into ONE category:

Sentence: "{sentence}"

Categories:
- wh-questions: Contains question words (what, who, where, when, why, how) or question structure
- passive: Contains passive voice (subject receives action, often with "by")
- complex-np: Contains complex noun phrases or embedded structures
- none: Simple declarative sentence with none of the above

Think: What is the main syntactic feature?

Classification:"""
    
    response = llm(prompt, max_new_tokens=5, do_sample=False)[0]['generated_text']
    response_clean = response.replace(prompt, "").strip().lower()
    
    # Parse response
    if 'wh' in response_clean:
        return 'wh_questions'
    elif 'passive' in response_clean:
        return 'passives'
    elif 'complex' in response_clean:
        return 'complex_np'
    else:
        return 'none'

def evaluate_dataset(dataset_name, expected_answer, name):
    print(f"\n=== {name.upper()} (expecting: {expected_answer}) ===")
    data = load_dataset("blimp", dataset_name)['train']['sentence_good']
    
    correct = 0
    total = 10
    
    for i, sentence in enumerate(data[:total]):
        llm_answer = ask_llm_reasoning(sentence)
        
        is_correct = llm_answer == expected_answer
        if is_correct:
            correct += 1
        
        print(f"{i+1:2d}. {sentence[:50]}...")
        print(f"    LLM: {llm_answer} | Expected: {expected_answer} | {'✓' if is_correct else '✗'}")
    
    accuracy = correct / total
    print(f"\nAccuracy: {accuracy:.3f}")
    return accuracy

# Evaluate
results = {}
for name, dataset_name in target_datasets.items():
    results[name] = evaluate_dataset(dataset_name, name, name)

results['control'] = evaluate_dataset(control_dataset, 'none', 'control')

print(f"\n{'='*50}")
print("FINAL RESULTS:")
for name, acc in results.items():
    print(f"{name:15}: {acc:.3f}")

Device set to use cpu



=== WH_QUESTIONS (expecting: wh_questions) ===


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


 1. Joel discovered the vase that Patricia might take....
    LLM: none | Expected: wh_questions | ✗
 2. Rodney thought about many paintings that Wendy had...
    LLM: none | Expected: wh_questions | ✗
 3. Susan wasn't thinking about the rivers that Kennet...
    LLM: none | Expected: wh_questions | ✗


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


...

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


 4. Jeffrey's sons are insulted by Tina's supervisor....
    LLM: none | Expected: passives | ✗
 5. Tracy isn't fired by Jodi's daughter....
    LLM: none | Expected: passives | ✗
 6. Eva's bosses weren't cared for by drivers....
    LLM: none | Expected: passives | ✗


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


 7. Some guys are disgusted by Angela....
    LLM: none | Expected: passives | ✗
 8. That guest's supervisor wasn't disliked by Wayne's...
    LLM: none | Expected: passives | ✗
 9. Curtis was distracted by these high schools....
    LLM: none | Expected: passives | ✗
10. David wasn't disgusted by these hospitals....
    LLM: none | Expected: passives | ✗

Accuracy: 0.000

=== COMPLEX_NP (expecting: complex_np) ===


ValueError: BuilderConfig 'complex_np_islands' not found. Available: ['adjunct_island', 'anaphor_gender_agreement', 'anaphor_number_agreement', 'animate_subject_passive', 'animate_subject_trans', 'causative', 'complex_NP_island', 'coordinate_structure_constraint_complex_left_branch', 'coordinate_structure_constraint_object_extraction', 'determiner_noun_agreement_1', 'determiner_noun_agreement_2', 'determiner_noun_agreement_irregular_1', 'determiner_noun_agreement_irregular_2', 'determiner_noun_agreement_with_adj_2', 'determiner_noun_agreement_with_adj_irregular_1', 'determiner_noun_agreement_with_adj_irregular_2', 'determiner_noun_agreement_with_adjective_1', 'distractor_agreement_relational_noun', 'distractor_agreement_relative_clause', 'drop_argument', 'ellipsis_n_bar_1', 'ellipsis_n_bar_2', 'existential_there_object_raising', 'existential_there_quantifiers_1', 'existential_there_quantifiers_2', 'existential_there_subject_raising', 'expletive_it_object_raising', 'inchoative', 'intran